# HTTM ThreeBehavior V2 - YOLO11s Kaggle

## TRAIN_READY_EVALUATION_LIMITED

Train and group-held-out Validation are available, so training can start without a Test split. Annotation completeness remains **UNVERIFIED**; Validation metrics are diagnostics and not an independent final evaluation. No Test is created or evaluated.

Upload the Kaggle-ready dataset ZIP as a Kaggle Input dataset. Set the accelerator to GPU for training. This notebook never writes to any local official checkpoint.


In [ ]:
from pathlib import Path
import json, zipfile, shutil, math, csv, os
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/httm_threebehavior_v2')
WORK.mkdir(parents=True, exist_ok=True)
EXPECTED_CLASSES = {'0': 'sleeping', '1': 'using_phone', '2': 'raising_hand'}
EXPECTED_COUNTS = {'train': 1086, 'val': 214}

def is_v2_readiness(obj):
    return (obj.get('classes') == EXPECTED_CLASSES
            and obj.get('image_counts', {}).get('train') == EXPECTED_COUNTS['train']
            and obj.get('image_counts', {}).get('val') == EXPECTED_COUNTS['val'])

def find_extracted_v2(root):
    if not root.exists(): return None
    for manifest in root.rglob('readiness.json'):
        try:
            obj = json.loads(manifest.read_text(encoding='utf-8'))
            if is_v2_readiness(obj) and (manifest.parent / 'data.yaml').is_file():
                return manifest.parent, obj
        except (OSError, ValueError):
            continue
    return None

found = find_extracted_v2(INPUT)
if found:
    DATA, READY = found
else:
    archive_paths = []
    if INPUT.exists():
        archive_paths = sorted(INPUT.rglob('*.zip'), key=lambda p: (p.name != 'HTTM_ThreeBehavior_V2_Kaggle.zip', str(p).lower()))
    selected_archive = None
    for archive in archive_paths:
        try:
            with zipfile.ZipFile(archive) as zf:
                for member in zf.namelist():
                    if Path(member).name.lower() == 'readiness.json':
                        obj = json.loads(zf.read(member).decode('utf-8'))
                        if is_v2_readiness(obj):
                            selected_archive = archive
                            break
            if selected_archive: break
        except (OSError, zipfile.BadZipFile, ValueError):
            continue
    if selected_archive:
        DATA_ROOT = WORK / 'dataset'
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(selected_archive) as zf:
            bad = zf.testzip()
            if bad: raise RuntimeError(f'ZIP CRC error: {bad}')
            zf.extractall(DATA_ROOT)
        found = find_extracted_v2(DATA_ROOT)
        if not found: raise RuntimeError(f'V2 archive did not contain the expected dataset structure: {selected_archive}')
        DATA, READY = found
    else:
        discovered = [str(p) for p in INPUT.rglob('data.yaml')][:20] if INPUT.exists() else []
        raise FileNotFoundError('Could not identify HTTM ThreeBehavior V2 by its readiness manifest (expected Train=1086, Val=214, classes sleeping/using_phone/raising_hand). Add the extracted V2 dataset as Kaggle Input. data.yaml files found: ' + repr(discovered))

print('dataset_path=', DATA)
print('run_status=TRAIN_READY_EVALUATION_LIMITED')
print('image_counts=', READY['image_counts'])
print('test_evaluation_ready=', READY['test_evaluation_ready'])


In [ ]:
import yaml
data_yaml = DATA / 'data.yaml'
cfg = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
assert cfg['nc'] == 3 and cfg['names'] == {0:'sleeping', 1:'using_phone', 2:'raising_hand'}, cfg
for split in ('train', 'val'):
    image_dir = DATA / cfg[split]
    label_dir = DATA / 'labels' / split
    images = [p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    labels = list(label_dir.glob('*.txt'))
    if not images or len(images) != len(labels):
        raise RuntimeError(f'{split}: image/label count mismatch or empty split')
    split_classes = set()
    GEOM_EPS = 1e-6  # tolerate floating-point rounding at normalized image borders
    for image in images:
        label = label_dir / (image.stem + '.txt')
        if not label.is_file() or not label.read_text(encoding='utf-8').strip():
            raise RuntimeError(f'{split}: missing/empty positive label for {image.name}')
        for line_no, line in enumerate(label.read_text(encoding='utf-8').splitlines(), 1):
            parts = line.split()
            if len(parts) != 5: raise RuntimeError(f'{label}:{line_no}: expected YOLO 5 columns')
            c = int(parts[0]); x,y,w,h = map(float, parts[1:])
            if c not in (0,1,2) or not all(math.isfinite(v) for v in (x,y,w,h)) or w<=0 or h<=0 or min(x-w/2,y-h/2)<-GEOM_EPS or max(x+w/2,y+h/2)>1+GEOM_EPS:
                raise RuntimeError(f'{label}:{line_no}: invalid class or bbox; class={c}, xywh={(x,y,w,h)}, bounds={(x-w/2,y-h/2,x+w/2,y+h/2)}')
            split_classes.add(c)
    if split_classes != {0,1,2}:
        raise RuntimeError(f'{split}: all target classes must be represented; found {split_classes}')
    print(split, 'images=',len(images), 'labels=',len(labels), 'classes=',sorted(split_classes))
if not READY.get('training_ready_with_diagnostic_val'):
    raise RuntimeError('Train/Validation are not marked ready for diagnostic training.')
if READY['image_counts'].get('train') != 1086 or READY['image_counts'].get('val') != 214:
    raise RuntimeError(f'Unexpected V2 split counts; refusing a different dataset: {READY["image_counts"]}')
if READY['test_evaluation_ready']:
    if not cfg.get('test') or not (DATA / cfg['test']).is_dir():
        raise RuntimeError('Readiness claims a Test split but data.yaml path is missing')
else:
    print('TEST_EVALUATION_BLOCKED: no credible independent Test set in this package.')


## Train YOLO11s ? TRAIN_READY_EVALUATION_LIMITED

Ultralytics uses Validation for checkpoint selection/early stopping. Batch starts at 16 and steps down only on a recognized CUDA out-of-memory error. Validation selects `best.pt`; annotation completeness remains UNVERIFIED. No Test split is required or synthesized.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('ultralytics') is None:
    print('Ultralytics is missing; installing it in this Kaggle session...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'ultralytics'])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Could not install ultralytics. Enable Internet in Kaggle Notebook Settings, then rerun this cell.') from exc
import torch
from ultralytics import YOLO
runtime_cfg = dict(cfg)
runtime_cfg['path'] = str(DATA.resolve())
runtime_yaml = WORK / 'data_runtime.yaml'
runtime_yaml.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding='utf-8')
DATA_YAML = str(runtime_yaml)
BATCH = 16
model = YOLO('yolo11s.pt')
while True:
    try:
        model.train(data=DATA_YAML, imgsz=640, epochs=100, batch=BATCH,
                    patience=25, seed=42, optimizer='auto', device=0,
                    project=str(WORK/'runs'), name=f'yolo11s_v2_batch{BATCH}',
                    exist_ok=False, pretrained=True, val=True)
        break
    except RuntimeError as exc:
        is_oom = 'out of memory' in str(exc).lower() or 'cuda error: out of memory' in str(exc).lower()
        if not is_oom or BATCH <= 2: raise
        BATCH //= 2
        print('CUDA OOM; retrying with batch=', BATCH)
        del model
        torch.cuda.empty_cache()
        model = YOLO('yolo11s.pt')
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights' / 'best.pt'
if not BEST.is_file(): raise RuntimeError(f'Ultralytics did not produce best.pt: {BEST}')
print('selected validation checkpoint=', BEST)
print('run_status=TRAIN_READY_EVALUATION_LIMITED')


In [ ]:
# Validation diagnostics after the checkpoint is selected.
selected = YOLO(str(BEST))
val_metrics = selected.val(data=DATA_YAML, split='val', imgsz=640, device=0, plots=True)
def per_class_metrics(metrics):
    box = metrics.box
    out = {}
    ids = list(getattr(box, 'ap_class_index', []))
    for i, class_id in enumerate(ids):
        name = cfg['names'][int(class_id)]
        out[name] = {
            'precision': float(box.p[i]), 'recall': float(box.r[i]),
            'mAP50': float(box.ap50[i]), 'mAP50_95': float(box.ap[i].mean())
        }
    return out
val_by_class = per_class_metrics(val_metrics)
print('UNVERIFIED_VALIDATION_DIAGNOSTICS=', json.dumps(val_by_class, indent=2))


In [ ]:
# Test runs only when a future dataset package supplies a credible Test split.
if READY['test_evaluation_ready']:
    test_metrics = selected.val(data=DATA_YAML, split='test', imgsz=640, device=0, plots=True)
    test_by_class = per_class_metrics(test_metrics)
    print('TEST_METRICS=', json.dumps(test_by_class, indent=2))
else:
    test_by_class = None
    print('TEST_SKIPPED: package status does not authorize Test evaluation.')


In [ ]:
# Package experiment artifacts; never touches local best.pt.
ART = WORK / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, ART / 'httm_three_behavior_v2_best.pt')
# Preserve Ultralytics training curves and Validation diagnostic plots.
for p in RUN_DIR.glob('*.png'):
    shutil.copy2(p, ART / p.name)
for p in Path(val_metrics.save_dir).glob('*.png'):
    shutil.copy2(p, ART / ('validation_' + p.name))
# Save a small, deterministic set of Validation predictions for visual inspection.
val_dir = DATA / cfg['val']
val_samples = sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}])[:24]
pred_dir = WORK / 'validation_prediction_samples'
selected.predict(source=[str(p) for p in val_samples], imgsz=640, device=0, save=True,
                 project=str(pred_dir), name='samples', exist_ok=True, verbose=False)
sample_dir = pred_dir / 'samples'
if sample_dir.exists():
    shutil.copytree(sample_dir, ART / 'validation_prediction_samples', dirs_exist_ok=True)
result = {'run_status': 'TRAIN_READY_EVALUATION_LIMITED', 'dataset_status': READY['status'],
          'annotation_completeness': 'UNVERIFIED', 'evaluation_status': 'VALIDATION_DIAGNOSTICS_ONLY_NO_TEST', 'batch_used': BATCH,
          'imgsz': 640, 'epochs': 100, 'patience': 25, 'seed': 42,
          'optimizer': 'auto', 'validation_metrics_unverified': val_by_class,
          'test_metrics': test_by_class, 'best_checkpoint': 'httm_three_behavior_v2_best.pt'}
(ART / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
artifact_zip = WORK / 'HTTM_ThreeBehavior_V2_training_artifacts.zip'
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in ART.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(ART).as_posix())
print('artifacts=', artifact_zip)
